# clue_rl

> Phase 12, Track B: RL on the clue board (GRPO with one shared adapter, or one per agent slot). Per step, new boards are each played `group` times with the current model; every slot's return (the reward scheme's, minus a training-only penalty for invalid replies) is compared with its group's; one policy-gradient update with a KL penalty to the warm-start model; checkpoints with exact resume (`rl_torch`). Phase 13 adds a training-only cost per wrong answer, per-turn advantages and the roll-back of an update after a jump in invalid replies (`rl_adv`, `rl_control`); at their defaults, runs and their hashes are Phase 12's. Training needs the `[train]` extra; the offline tests do not.

In [ ]:
#| default_exp clue_rl

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib
import json
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np

from explRL.clue_grid import BOARD, CLUE_SETTINGS
from explRL.clue_roll import coopRows, rollout
from explRL.clue_sft import libVersions, loadModel, loraConfig
from explRL.llm_runs import LLMConfig
from explRL.local_policy import pickDevice
from explRL.rl_adv import groupAdvantages, stepSamples, trainingReturns, turnAdvantages
from explRL.rl_control import (haltReason, invalidShare, jumpGuard, microBatches, nanMean, nextLr, progressBar, stepLine,
                               warmLr)
from explRL.rl_torch import (freeGPU, loadCheckpoint, makePolicies, policyUpdate, probeKL, probeLogps, restore, saveCheckpoint,
                             saveMemory, snapshot)


@dataclass
class RLConfig:
    model: str                  # the warm-start model (merged weights)
    out: str
    scheme: str = 'credit'
    N: int = 4
    channel: str = 'blackboard'
    turns: int = 10
    seedsPerStep: int = 8
    group: int = 4
    steps: int = 40
    lr: float = 1e-5
    kl: float = 0.04
    loraR: int = 16
    loraAlpha: int = 32
    maxNew: int = 160
    temperature: float = 1.0
    formatPenalty: float = 0.1
    trainSeed: int = 0
    boardSeed0: int = 10_000
    ckptEvery: int = 5
    microBatch: int = 8
    maxLen: int = 1536
    device: str = 'auto'
    perRole: bool = False       # one adapter per agent slot (the optional ablation)
    gradCkpt: bool = True       # gradient checkpointing (less memory, about a third more compute)
    turnFraction: float = 0.25  # train on this share (seeded per step) of the turns with a nonzero advantage
    haltInvalid: float = 0.10   # stopping rule: invalid-reply rate above this for haltRun steps in a row ...
    haltRun: int = 3
    haltKL: float = 0.5         # ... or the mean KL above this
    klTarget: float = 0.0       # per-step policy change the learning rate is steered to (0: fixed learning rate)
    lrMin: float = 1e-6
    lrMax: float = 3e-4
    warmup: int = 0             # steps of linear learning-rate warm-up
    rejectFactor: float = 4.0   # a step whose change exceeds rejectFactor x klTarget is undone
    probe: int = 32             # replies re-scored after each update to measure its policy change
    advantage: str = 'board'    # Phase 13: 'board' (one per board and slot) or 'turn' (per text turn, from the reward-to-go)
    wrongPenalty: float = 0.0   # training-only extra cost per wrong answer (the board keeps its own penalty)
    haltJump: float = None      # roll an update back when the next step's invalid share is above this (None: off)


RL13_DEFAULTS = dict(advantage='board', wrongPenalty=0.0, haltJump=None)


def runFields(cfg):
    "The configuration as saved and hashed: Phase 13's fields only where they differ from their defaults (Phase 12 runs unchanged)."
    return {k: v for k, v in asdict(cfg).items() if k not in RL13_DEFAULTS or v != RL13_DEFAULTS[k]}


def configHash(cfg):
    "Identity of a training run: everything but where it runs and how long (a run can be resumed and extended)."
    d = {k: v for k, v in runFields(cfg).items() if k not in ('out', 'device', 'ckptEvery', 'microBatch', 'steps', 'gradCkpt')}
    return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:12]


def adapterNames(cfg):
    return [f'slot{i}' for i in range(cfg.N)] if cfg.perRole else ['default']


def stepSeeds(cfg, s):
    "Board seeds of step s: seedsPerStep new boards, each played `group` times."
    return [cfg.boardSeed0 + s * cfg.seedsPerStep + j for j in range(cfg.seedsPerStep) for _ in range(cfg.group)]


def trainRL(cfg, settings=CLUE_SETTINGS, show=True):
    "GRPO on the clue board from the warm-start model (one update per step, KL to the adapter-free model); -> final adapter."
    import torch
    from peft import get_peft_model
    assert cfg.advantage in ('board', 'turn'), cfg.advantage
    out, dev = Path(cfg.out), pickDevice(cfg.device)
    out.mkdir(parents=True, exist_ok=True)
    (out / 'config.json').write_text(json.dumps(dict(runFields(cfg), hash=configHash(cfg), versions=libVersions()), indent=1) + '\n')
    torch.manual_seed(cfg.trainSeed)
    tok, base = loadModel(cfg.model, dev)
    names = adapterNames(cfg)
    model = get_peft_model(base, loraConfig(cfg.loraR, cfg.loraAlpha), adapter_name=names[0])
    for n in names[1:]:
        model.add_adapter(n, loraConfig(cfg.loraR, cfg.loraAlpha))
    saveMemory(model, cfg.gradCkpt)
    params = [p for n, p in model.named_parameters() if 'lora_' in n]
    for p in params:
        p.requires_grad_(True)
    opt = torch.optim.AdamW(params, lr=cfg.lr)
    start = loadCheckpoint(model, opt, out, configHash(cfg), names)
    if (out / 'halted.json').exists():                       # stopped by the stopping rule: never silently resumed
        print('halted:', json.loads((out / 'halted.json').read_text())['reason'])
        return str(out / 'final')
    mp = out / 'metrics.jsonl'                               # on resume, drop the steps after the checkpoint (redone)
    history = [h for h in map(json.loads, open(mp)) if h['step'] <= start] if mp.exists() else []
    mp.write_text(''.join(json.dumps(h) + '\n' for h in history))
    lr = history[-1].get('lrNext', cfg.lr) if history else cfg.lr
    views, pol, vllm = makePolicies(cfg, model, tok, names, dev)
    lcfg = LLMConfig(task='clue', channel=cfg.channel, N=cfg.N, turns=cfg.turns, model=cfg.model,
                     world={**{k: settings[k] for k in BOARD}, 'scheme': cfg.scheme})
    bar = progressBar(cfg.steps, start, f'{cfg.scheme} RL', show)   # with one line per step below it
    kept, rolls = None, 0                                    # with haltJump: the last update's snapshot; roll-backs in a row
    with open(out / 'metrics.jsonl', 'a') as log:
        for s in range(start, cfg.steps):
            t0 = time.time()

            def play():
                model.eval()
                if vllm:
                    model.save_pretrained(out / 'live')
                for v, n in zip(views, names):
                    v.calls = s * 100_000                   # sampling seeds depend on the step only (exact resume)
                    if vllm:
                        v.setAdapter(str(out / 'live' / ('' if n == 'default' else n)))
                return rollout(lcfg, stepSeeds(cfg, s), pol, keepPrompts=True)

            jumped = lambda ro: kept is not None and invalidShare(coopRows(lcfg, ro)) > cfg.haltJump
            ro, back, rolls = jumpGuard(play, jumped, lambda: restore(params, opt, kept), rolls)
            lr *= 0.5 ** back                                # halved per roll-back (FR-D4)
            if ro is None:
                reason = f'jump: invalid replies above {cfg.haltJump:g} after two roll-backs in a row (step {s + 1})'
                (out / 'halted.json').write_text(json.dumps(dict(step=s, reason=reason)) + '\n')
                bar.write(f'halted: {reason}')
                break
            R = trainingReturns(ro, cfg.formatPenalty, cfg.wrongPenalty)
            adv = (turnAdvantages(ro, cfg.formatPenalty, cfg.wrongPenalty) if cfg.advantage == 'turn' else
                   groupAdvantages(R.reshape(cfg.seedsPerStep, cfg.group, cfg.N)).reshape(R.shape))
            signal = stepSamples(ro, adv)
            samples = stepSamples(ro, adv, cfg.turnFraction, np.random.default_rng([cfg.trainSeed, s]))
            t1 = time.time()
            model.train()
            for g in opt.param_groups:
                g['lr'] = warmLr(lr, s, cfg.warmup)
            probe = microBatches(samples[::max(1, len(samples) // cfg.probe)][:cfg.probe], cfg)
            before = probeLogps(model, tok, probe, names, cfg.perRole, cfg.maxLen, dev)
            snap = snapshot(params, opt) if cfg.klTarget or cfg.haltJump is not None else None
            stats = policyUpdate(model, tok, opt, params, samples, microBatches(samples, cfg), names, cfg.perRole, cfg.kl,
                                 cfg.maxLen, dev)
            stepKL = probeKL(before, probeLogps(model, tok, probe, names, cfg.perRole, cfg.maxLen, dev))
            lrNext, rejected = nextLr(lr, stepKL, cfg, warming=s + 1 < cfg.warmup, trained=bool(samples))
            if rejected:                                     # a runaway step is undone (weights and optimizer)
                restore(params, opt, snap)
            kept = snap if cfg.haltJump is not None else None
            rows = coopRows(lcfg, ro)
            m = dict(step=s + 1, rolloutSeconds=round(t1 - t0, 2), updateSeconds=round(time.time() - t1, 2),
                     meanReturn=float(R.mean()), welfare=float(np.mean([r['welfare'] for r in rows])),
                     sharePerHeld=float(np.mean([r['sharePerHeld'] for r in rows])),
                     invalid=invalidShare(rows),
                     soloShare=nanMean([r['soloShare'] for r in rows]), signalTurns=len(signal), samples=len(samples),
                     lr=warmLr(lr, s, cfg.warmup), stepKL=stepKL, rejected=rejected, lrNext=lrNext,
                     gradCkpt=bool(getattr(model, 'is_gradient_checkpointing', False)), **stats,
                     **({} if cfg.haltJump is None else dict(rolledBack=back)))
            log.write(json.dumps(m) + '\n')
            log.flush()
            history.append(m)
            bar.write(stepLine(m, cfg.steps)) if show else None
            bar.set_postfix(lr=f"{m['lr']:.2g}", stepKL=f"{stepKL:.4f}", welfare=f"{m['welfare']:.3f}")
            bar.update(1)
            lr = lrNext
            reason = haltReason(history, cfg)
            if (s + 1) % cfg.ckptEvery == 0 or s + 1 == cfg.steps or reason:
                saveCheckpoint(model, opt, s + 1, out, configHash(cfg), ro)
            if reason:
                (out / 'halted.json').write_text(json.dumps(dict(step=s + 1, reason=reason)) + '\n')
                bar.write(f'halted: {reason}')             # through the bar, so a live bar is not garbled
                break
    bar.close()
    model.save_pretrained(out / 'final')
    del model, base, opt, params
    freeGPU(*views)                                          # vLLM runs in its own process: free its memory too
    return str(out / 'final')

## Tests

In [ ]:
# returns, advantages and samples are tested in rl_adv; the loop's control logic in rl_control
# seeds: new boards every step, each played `group` times; configuration identity
c = RLConfig('m', 'o', seedsPerStep=2, group=3)
assert stepSeeds(c, 0) == [10000] * 3 + [10001] * 3 and stepSeeds(c, 1)[0] == 10002
assert configHash(c) == configHash(RLConfig('m', 'elsewhere', seedsPerStep=2, group=3, device='cpu', steps=80, gradCkpt=False))
assert configHash(c) != configHash(RLConfig('m', 'o', seedsPerStep=2, group=3, lr=2e-5))
# NFR-1: Phase 13's fields at their defaults leave a run's identity and saved configuration as Phase 12's: the v2 runs' hashes
# (computed before the fields existed, and saved in their config.json) are unchanged
from explRL.coop_grid import RL_PLAN
P12 = dict(individual='542eb4f99c29', team='be13d2c8afb3', credit='c02fa4f72378')
v2 = lambda sc, **kw: RLConfig('/content/drive/MyDrive/explRL_rl/warm_model', f'/content/drive/MyDrive/explRL_rl/runs_v2/{sc}',
                               scheme=sc, **{**RL_PLAN['rl'], **kw})
assert {sc: configHash(v2(sc)) for sc in P12} == P12 and configHash(RLConfig('m', 'o')) == '087f167412ae'
assert configHash(v2('credit', advantage='board', wrongPenalty=0, haltJump=None)) == P12['credit']
assert set(runFields(v2('credit'))) == set(asdict(v2('credit'))) - set(RL13_DEFAULTS)            # config.json as Phase 12's
# each new field, set, is part of the identity; another training seed is another run (the credit control's seed 2)
assert len({configHash(v2('credit', **kw)) for kw in (dict(), dict(advantage='turn'), dict(wrongPenalty=2.5), dict(haltJump=0.15),
                                                     dict(trainSeed=1))}) == 5
assert configHash(v2('credit', trainSeed=1)) == 'd93b36d7c732' and runFields(v2('indpen', wrongPenalty=2.5))['wrongPenalty'] == 2.5
# per-role adapters: one per slot
assert adapterNames(RLConfig('m', 'o')) == ['default'] and adapterNames(RLConfig('m', 'o', perRole=True)) == ['slot0', 'slot1', 'slot2', 'slot3']

In [ ]:
#| train
# With the [train] extra, no download: two RL steps on the tiny random model (whose replies are all invalid, so random
# returns stand in to give nonzero advantages), then resume at step 3, a refused configuration, and per-role adapters
import os
if os.environ.get('IN_TEST'):   # under nbdev-test only: the docs build runs cells with imports and has no [train] extra
    import shutil
    import tempfile
    from pathlib import Path
    from explRL.tiny_model import tinyModel
    _rng, _tr = np.random.default_rng(0), trainingReturns   # trainRL here looks it up in this namespace
    trainingReturns = lambda ro, *a: _tr(ro, *a) + _rng.normal(size=np.shape(ro.returns))
    try:
        with tempfile.TemporaryDirectory() as d:
            tiny = tinyModel(d + '/tiny')
            for perRole in (False, True):
                cfg = RLConfig(tiny, f'{d}/rl{perRole}', steps=2, seedsPerStep=2, group=3, turns=3, ckptEvery=1, maxNew=24, perRole=perRole, lr=1e-3)
                trainRL(cfg)
                trainRL(RLConfig(**{**asdict(cfg), 'steps': 3}))
                m = [json.loads(x) for x in open(f'{d}/rl{perRole}/metrics.jsonl')]
                assert [x['step'] for x in m] == [1, 2, 3] and all(x['samples'] > 0 and np.isfinite(x['loss']) for x in m), m
                assert m[0]['kl'] == 0 and m[-1]['kl'] > 0                    # the fresh adapter starts at the warm-start model
                assert sorted(p.name for p in Path(f'{d}/rl{perRole}/turns').iterdir()) == ['step-0001.jsonl', 'step-0002.jsonl', 'step-0003.jsonl']
                assert json.load(open(f'{d}/rl{perRole}/config.json'))['versions']['torch']
                try:
                    trainRL(RLConfig(**{**asdict(cfg), 'lr': 2e-3, 'steps': 4}))
                    raise AssertionError('a checkpoint of another configuration must be refused')
                except ValueError:
                    pass
                # the tiny model's replies are all invalid: the stopping rule halts at step 3, and a halted run is not resumed
                assert 'invalid replies' in json.load(open(f'{d}/rl{perRole}/halted.json'))['reason']
                trainRL(RLConfig(**{**asdict(cfg), 'steps': 5}))
                assert len(open(f'{d}/rl{perRole}/metrics.jsonl').readlines()) == 3
            # a disconnect after step 3 but before its checkpoint: the redone step is logged once, not twice
            cfg = RLConfig(tiny, f'{d}/resume', steps=3, seedsPerStep=2, group=3, turns=3, ckptEvery=2, maxNew=24, lr=1e-3, haltRun=99)
            trainRL(cfg)
            shutil.rmtree(f'{d}/resume/ckpt-0003')
            trainRL(cfg)
            assert [json.loads(x)['step'] for x in open(f'{d}/resume/metrics.jsonl')] == [1, 2, 3]
            # the KL-targeted learning rate: with a target no step can meet, every step is undone and the adapter stays at zero
            from safetensors.torch import load_file
            cfg = RLConfig(tiny, f'{d}/ctl', steps=2, seedsPerStep=2, group=3, turns=3, ckptEvery=2, maxNew=24, lr=1e-3,
                           klTarget=1e-12, rejectFactor=1.0, haltRun=99)
            trainRL(cfg)
            m = [json.loads(x) for x in open(f'{d}/ctl/metrics.jsonl')]
            assert all(x['rejected'] and x['stepKL'] > 0 for x in m) and m[1]['lr'] == m[0]['lrNext'] < m[0]['lr'], m
            assert all(not v.any() for k, v in load_file(f'{d}/ctl/final/adapter_model.safetensors').items() if 'lora_B' in k)
            # Phase 13: per-turn advantages (the real ones, plus noise, as the tiny model's turns all tie) and the wrong-answer
            # penalty: two steps, then resume at step 3; no roll-back key without haltJump
            _ta = turnAdvantages
            turnAdvantages = lambda ro, *a: [[list(np.add(x, _rng.normal(size=len(x)))) for x in b] for b in _ta(ro, *a)]
            cfg = RLConfig(tiny, f'{d}/turn', steps=2, seedsPerStep=2, group=3, turns=3, ckptEvery=1, maxNew=24, lr=1e-3, haltRun=99,
                           advantage='turn', wrongPenalty=2.5)
            trainRL(cfg)
            trainRL(RLConfig(**{**asdict(cfg), 'steps': 3}))
            m = [json.loads(x) for x in open(f'{d}/turn/metrics.jsonl')]
            assert [x['step'] for x in m] == [1, 2, 3] and all(x['samples'] > 0 for x in m) and 'rolledBack' not in m[0], m
            assert json.load(open(f'{d}/turn/config.json'))['advantage'] == 'turn' and 'haltJump' not in json.load(open(f'{d}/turn/config.json'))
            # a forced jump (stub invalid shares, in call order: the metrics of step 1, then step 2's rollouts 0.9 (rolled back:
            # weights and optimizer restored, learning rate halved, the step played again) and 0.0 ...), then resume
            _is, shares = invalidShare, iter([0.0, 0.9, 0.0] + [0.0] * 20)
            invalidShare = lambda rows: next(shares)
            cfg = RLConfig(tiny, f'{d}/jump', steps=3, seedsPerStep=2, group=3, turns=3, ckptEvery=1, maxNew=24, lr=1e-3, haltRun=99,
                           haltJump=0.15)
            trainRL(cfg)
            m = [json.loads(x) for x in open(f'{d}/jump/metrics.jsonl')]
            assert [x['rolledBack'] for x in m] == [0, 1, 0] and m[1]['lr'] == m[0]['lrNext'] / 2 and m[2]['lr'] == m[1]['lrNext'], m
            trainRL(RLConfig(**{**asdict(cfg), 'steps': 4}))
            assert [json.loads(x)['step'] for x in open(f'{d}/jump/metrics.jsonl')] == [1, 2, 3, 4]
            invalidShare = _is
            # the real invalid share (the tiny model's replies are all invalid): step 2 jumps, is rolled back, jumps again: halt;
            # the final adapter is the one before step 1's update (zero)
            cfg = RLConfig(tiny, f'{d}/halt', steps=3, seedsPerStep=2, group=3, turns=3, ckptEvery=1, maxNew=24, lr=1e-3, haltRun=99,
                           haltJump=0.15)
            trainRL(cfg)
            assert [json.loads(x)['step'] for x in open(f'{d}/halt/metrics.jsonl')] == [1]
            assert json.load(open(f'{d}/halt/halted.json'))['reason'].startswith('jump')
            assert all(not v.any() for k, v in load_file(f'{d}/halt/final/adapter_model.safetensors').items() if 'lora_B' in k)
    finally:
        trainingReturns = _tr
        turnAdvantages = globals().get('_ta', turnAdvantages)
        invalidShare = globals().get('_is', invalidShare)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()